# 1. 방송 이벤트 정리 & 분석 설계

**목적**: 방송 일정, 촬영지, 시청률을 정리하고 분석 가능 구간을 식별한다.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from config import BROADCAST_DIR, BROADCAST_DONG_MAP, read_csv_auto

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

## 1.1 방송 목록 로드

In [ ]:
broadcast = read_csv_auto(BROADCAST_DIR / 'broadcast_data.csv')
locations = read_csv_auto(BROADCAST_DIR / 'broadcast_locations.csv')
budget = read_csv_auto(BROADCAST_DIR / 'broadcast_budget_summary.csv')
effects = read_csv_auto(BROADCAST_DIR / 'broadcast_effects_2025.csv')

print(f"방송 {len(broadcast)}건, 촬영지 {len(locations)}건")
broadcast[['연도', '프로그램명', '방송사', '방영일', '시청률_pct', '촬영지']].fillna('-')

## 1.2 방송 타임라인 시각화

In [ ]:
# 방영일 파싱 (확정된 것만)
timeline = broadcast[broadcast['방영일'].notna()].copy()

def parse_air_date(s):
    """방영일 문자열에서 첫 날짜 추출"""
    s = str(s).strip()
    for fmt in ['%Y-%m-%d', '%Y-%m-%d~%d']:
        try:
            return pd.to_datetime(s[:10])
        except:
            continue
    # 2026-06~07 같은 경우
    if len(s) >= 7 and s[4] == '-':
        try:
            return pd.to_datetime(s[:7] + '-01')
        except:
            pass
    return pd.NaT

timeline['air_date'] = timeline['방영일'].apply(parse_air_date)
timeline = timeline.dropna(subset=['air_date']).sort_values('air_date')

# 타임라인 차트
fig, ax = plt.subplots(figsize=(14, 5))
colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c', '#e67e22', '#34495e', '#7f8c8d']

for i, (_, row) in enumerate(timeline.iterrows()):
    ax.axvline(row['air_date'], color=colors[i % len(colors)], alpha=0.7, linewidth=2)
    rating = f" ({row['시청률_pct']}%)" if pd.notna(row['시청률_pct']) else ""
    ax.text(row['air_date'], len(timeline) - i - 0.5,
            f"{row['프로그램명']}{rating}",
            fontsize=9, ha='left', va='center',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=colors[i % len(colors)]))

# 데이터 커버리지 표시
data_ranges = {
    'BC카드 (주간)': ('2025-01-01', '2026-04-30'),
    'T맵 내비': ('2025-01-01', '2026-04-30'),
    'SKT 유입인구': ('2026-01-01', '2026-02-28'),
}
for j, (name, (s, e)) in enumerate(data_ranges.items()):
    ax.axhspan(-1.5 - j*0.6, -1.0 - j*0.6,
               xmin=(pd.Timestamp(s) - timeline['air_date'].min()).days / (timeline['air_date'].max() - timeline['air_date'].min()).days,
               xmax=(pd.Timestamp(e) - timeline['air_date'].min()).days / (timeline['air_date'].max() - timeline['air_date'].min()).days,
               alpha=0.3, color=colors[j])
    ax.text(pd.Timestamp(s), -1.25 - j*0.6, name, fontsize=8, va='center')

ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=45)
ax.set_title('아산시 방송 타임라인 & 데이터 커버리지', fontsize=13)
ax.set_yticks([])
plt.tight_layout()
plt.show()

## 1.3 방송별 촬영 읍면동 정리

In [ ]:
# 촬영지 → 읍면동 매핑
loc_summary = locations.groupby('프로그램명').agg(
    촬영지수=('촬영지명', 'count'),
    읍면동목록=('읍면동', lambda x: ', '.join(sorted(x.dropna().unique()))),
    관광지유형=('관광지유형', lambda x: ', '.join(sorted(x.dropna().unique()))),
).reset_index()

display(loc_summary)

## 1.4 분석 구간 설계

### 방송 간 간격 문제
- 2025-11월에 4개 방송 집중 → 개별 분리 어려움
- **그룹핑 전략**: 충분히 떨어진 방송만 개별 분석, 나머지는 묶어서 분석

In [ ]:
# 방송 간 간격 계산
t = timeline[['프로그램명', 'air_date']].sort_values('air_date').reset_index(drop=True)
t['다음방송까지_일'] = t['air_date'].diff().dt.days
t['이전방송'] = t['프로그램명'].shift(1)
display(t)

print("\n[분석 그룹 제안]")
print("그룹A (단독): 전국노래자랑 (2025-06-08) - 전후 5개월 여유")
print("그룹B (묶음): 11월 캠페인 (전현무계획2 + 굿모닝대한민국 + 6시내고향 + 같이삽시다) - 2025-11~12")
print("그룹C (단독): 뛰어야산다2 (2026-01-12) - SKT 데이터 활용 가능")
print("그룹D (단독): 황제파워 (2026-05-09) - BC카드 데이터 없음 (04월까지)")

## 1.5 기존 효과 분석 결과 (참고)

아산시에서 이미 측정한 '같이삽시다' 효과:

In [ ]:
if len(effects) > 0:
    display(effects)
else:
    print("기존 효과 데이터 없음")